![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)
# Fraud Detection in Financial Services with Redis & RedisVL

## Let's Begin!
<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/fraud-detection/00_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Card fraud is fundamentally a *similarity* problem: a fraudulent transaction
tends to look like other fraudulent transactions, and unlike the normal
spending behavior of the account it hits. That makes it a natural fit for
**vector search**.

In this notebook we use Redis as a real-time fraud engine:

1. Turn each transaction into a **feature vector** (amount, time, location,
   velocity, channel, …).
2. Index those vectors in Redis with **RedisVL**.
3. Score a new transaction three complementary ways:
   - **KNN fraud scoring** — how fraudulent do the most similar past
     transactions look?
   - **Anomaly detection** — how far is this transaction from the account's
     *normal* behavior?
   - **Velocity / rules** — combine vector similarity with metadata filters
     (card, merchant category, amount, time window) the way a real fraud
     system does.

We use engineered numeric features (not text embeddings) because fraud signals
are tabular — this also keeps the notebook fast, deterministic, and free of any
API keys.

## Packages

In [1]:
%pip install -q "redisvl>=0.11.0" pandas numpy scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Install Redis Stack

This notebook stores and indexes transaction feature vectors in Redis, so we
need a Redis instance with the search & query capability available.

#### For Colab
Use the shell script below to download, extract, and install [Redis Stack](https://redis.io/docs/getting-started/install-stack/) directly from the Redis package archive.

In [ ]:
# NBVAL_SKIP
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

#### For Alternative Environments
There are many ways to get the necessary redis-stack instance running
1. On cloud, deploy a [FREE instance of Redis in the cloud](https://redis.com/try-free/). Or, if you have your own version of Redis Enterprise running, that works too!
2. Per OS, [see the docs](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/)
3. With docker: `docker run -d --name redis-stack-server -p 6379:6379 redis/redis-stack-server:latest`

### Define the Redis Connection URL

By default this notebook connects to the local instance of Redis Stack. **If you have your own Redis Enterprise instance** - replace REDIS_PASSWORD, REDIS_HOST and REDIS_PORT values with your own.

In [2]:
import os
import warnings

warnings.filterwarnings('ignore')

# Replace values below with your own if using Redis Cloud instance
REDIS_HOST = os.getenv("REDIS_HOST", "localhost") # ex: "redis-18374.c253.us-central1-1.gce.cloud.redislabs.com"
REDIS_PORT = os.getenv("REDIS_PORT", "6379")      # ex: 18374
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")  # ex: "1TNxTEdYRDgIDKM2gDfasupCADXXXX"

# If SSL is enabled on the endpoint, use rediss:// as the URL prefix
REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

### Create redis client

In [3]:
from redis import Redis

client = Redis.from_url(REDIS_URL)
client.ping()

True

## Generate a synthetic transaction dataset

Real card data is sensitive and rarely shareable, so we simulate a labeled
dataset that captures the signals fraud teams actually use. Each transaction
has both **raw metadata** (card id, merchant category, amount, country,
timestamp) and **behavioral features** used for the vector:

| feature | fraud signal |
|---|---|
| `amount` | fraud skews toward large or oddly-round amounts |
| `hour` | fraud clusters in the middle of the night |
| `distance_from_home` | card-present fraud happens far from the cardholder |
| `distance_from_last_txn` | impossible travel between consecutive swipes |
| `ratio_to_median_amount` | spend far above the account's typical purchase |
| `num_txn_last_hour` | velocity — many transactions in a short window |
| `is_online` | online / card-not-present is higher risk |

Roughly 3% of the transactions are fraudulent, similar to a stressed real-world
mix.

In [4]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N = 5000
FRAUD_RATE = 0.03
n_fraud = int(N * FRAUD_RATE)
n_legit = N - n_fraud

CATEGORIES = ["grocery", "restaurant", "travel", "electronics", "fuel", "online_retail", "atm_withdrawal"]
COUNTRIES = ["US", "CA", "GB", "DE", "FR", "NG", "RU", "BR"]


def make_transactions(n, fraud):
    if fraud:
        amount = rng.lognormal(mean=6.0, sigma=1.0, size=n)          # larger spend
        hour = rng.choice(range(24), size=n, p=_night_heavy())       # late night
        distance_from_home = rng.exponential(scale=400, size=n)      # far away
        distance_from_last = rng.exponential(scale=300, size=n)      # impossible travel
        ratio_to_median = rng.lognormal(mean=1.2, sigma=0.6, size=n) # well above normal
        num_txn_last_hour = rng.poisson(lam=4.0, size=n) + 1         # bursty velocity
        is_online = rng.binomial(1, 0.7, size=n)
        country = rng.choice(COUNTRIES, size=n, p=[0.25,0.05,0.1,0.05,0.05,0.2,0.2,0.1])
    else:
        amount = rng.lognormal(mean=3.5, sigma=0.8, size=n)
        hour = rng.choice(range(24), size=n, p=_day_heavy())
        distance_from_home = rng.exponential(scale=25, size=n)
        distance_from_last = rng.exponential(scale=20, size=n)
        ratio_to_median = rng.lognormal(mean=0.0, sigma=0.3, size=n)
        num_txn_last_hour = rng.poisson(lam=0.5, size=n) + 1
        is_online = rng.binomial(1, 0.3, size=n)
        country = rng.choice(COUNTRIES, size=n, p=[0.6,0.1,0.1,0.05,0.05,0.03,0.02,0.05])

    return pd.DataFrame({
        "amount": np.round(amount, 2),
        "hour": hour,
        "distance_from_home": np.round(distance_from_home, 1),
        "distance_from_last_txn": np.round(distance_from_last, 1),
        "ratio_to_median_amount": np.round(ratio_to_median, 3),
        "num_txn_last_hour": num_txn_last_hour,
        "is_online": is_online,
        "merchant_category": rng.choice(CATEGORIES, size=n),
        "country": country,
        "is_fraud": int(fraud),
    })


def _night_heavy():
    """ bias toward nighttime transactions """
    w = np.array([3,3,3,3,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,3,3,3], dtype=float)
    return w / w.sum()


def _day_heavy():
    """ bias toward daytime transactions """
    w = np.array([1,1,1,1,1,1,2,3,4,4,4,5,5,4,4,4,4,4,5,4,3,2,1,1], dtype=float)
    return w / w.sum()


df = pd.concat([make_transactions(n_legit, False), make_transactions(n_fraud, True)], ignore_index=True)
df = df.sample(frac=1.0, random_state=1).reset_index(drop=True) # shuffle the legit and fraud rows together

# create identifiers + a synthetic event time (seconds since an arbitrary epoch)
df["card_id"] = ["card_" + str(i) for i in rng.integers(0, 800, size=len(df))]
df["txn_id"] = ["txn_" + str(i) for i in range(len(df))]
df["timestamp"] = rng.integers(1_700_000_000, 1_700_000_000 + 60*60*24*30, size=len(df))

# Inject one account-takeover burst: 12 transactions on a single card within
# a 1-hour window. This is the classic "velocity" fraud pattern we detect later.
BURST_CARD = "card_burst"
BURST_START = 1_700_500_000
burst = make_transactions(12, fraud=True)
burst["card_id"] = BURST_CARD
burst["txn_id"] = ["txn_burst_" + str(i) for i in range(len(burst))]
burst["timestamp"] = BURST_START + rng.integers(0, 60 * 60, size=len(burst))  # within one hour
df = pd.concat([df, burst], ignore_index=True)

print(f"{len(df)} transactions, {df.is_fraud.sum()} fraudulent ({df.is_fraud.mean():.1%})")
df.head()

5012 transactions, 162 fraudulent (3.2%)


,amount,hour,distance_from_home,distance_from_last_txn,ratio_to_median_amount,num_txn_last_hour,is_online,merchant_category,country,is_fraud,card_id,txn_id,timestamp
0,21.46,11,31.0,2.3,0.813,1,0,travel,US,0,card_719,txn_0,1701728087
1,77.31,15,160.9,2.6,1.175,1,0,electronics,GB,0,card_669,txn_1,1701272998
2,83.01,9,41.1,48.8,0.960,1,1,online_retail,BR,0,card_651,txn_2,1701690135
3,53.29,12,47.2,7.3,1.031,1,0,electronics,CA,0,card_258,txn_3,1701737612
4,23.99,8,8.3,11.2,0.953,2,0,travel,FR,0,card_571,txn_4,1701438900


### Build the feature vector

We standardize the seven behavioral features (zero mean, unit variance) and pack
them into one vector per transaction. Standardizing matters: without it,
`amount` (hundreds) would dominate `is_online` (0/1) purely because of scale.

We split the data into a **reference set** (what we index and search against) and
a small **test set** of unseen transactions to score later. In production the
reference set is your history of labeled transactions.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

FEATURES = [
    "amount", "hour", "distance_from_home", "distance_from_last_txn",
    "ratio_to_median_amount", "num_txn_last_hour", "is_online",
]
VECTOR_DIM = len(FEATURES)

# keep the injected burst out of the test split so the velocity demo can find it
splittable = df[df["card_id"] != "card_burst"]
ref_df, test_df = train_test_split(splittable, test_size=200, random_state=7, stratify=splittable["is_fraud"])
ref_df = pd.concat([ref_df, df[df["card_id"] == "card_burst"]]).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# fit the scaler ONLY on reference data, then apply to both
scaler = StandardScaler().fit(ref_df[FEATURES])

def to_vectors(frame):
    return scaler.transform(frame[FEATURES]).astype(np.float32)

ref_vectors = to_vectors(ref_df)
test_vectors = to_vectors(test_df)

ref_df["vector"] = list(ref_vectors)
print("vector dim:", VECTOR_DIM, "| reference:", len(ref_df), "| test:", len(test_df))

vector dim: 7 | reference: 4812 | test: 200


## Define the Redis index schema

We index the raw transaction fields (so we can filter and investigate) plus the
feature `vector`. We use an **HNSW** vector index with **L2 (Euclidean)**
distance — for standardized feature vectors, L2 directly measures how different
two transactions' behaviors are.

In [6]:
from redisvl.schema import IndexSchema
from redisvl.index import SearchIndex

index_name = "transactions"

schema = IndexSchema.from_dict({
    "index": {
        "name": index_name,
        "prefix": index_name,
        "storage_type": "hash",
    },
    "fields": [
        {"name": "txn_id", "type": "tag"},
        {"name": "card_id", "type": "tag"},
        {"name": "merchant_category", "type": "tag", "attrs": {"sortable": True}},
        {"name": "country", "type": "tag", "attrs": {"sortable": True}},
        {"name": "amount", "type": "numeric", "attrs": {"sortable": True}},
        {"name": "timestamp", "type": "numeric", "attrs": {"sortable": True}},
        {"name": "is_fraud", "type": "numeric", "attrs": {"sortable": True}},
        {
            "name": "vector",
            "type": "vector",
            "attrs": {
                "dims": VECTOR_DIM,
                "distance_metric": "l2",
                "algorithm": "hnsw",
                "datatype": "float32",
            },
        },
    ],
})

index = SearchIndex(schema, client)
index.create(overwrite=True, drop=True)

In [7]:
!rvl index info -i transactions -u {REDIS_URL}



Index Information:
╭──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────╮
│ Index Name       │ Storage Type     │ Prefixes         │ Index Options    │ Indexing         │
├──────────────────┼──────────────────┼──────────────────┼──────────────────┼──────────────────┤
| transactions     | HASH             | ['transactions'] | []               | 0                |
╰──────────────────┴──────────────────┴──────────────────┴──────────────────┴──────────────────╯
Index Fields:
╭───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────╮
│ Name              │ Attribute         │ Type              │ Field Option      │ Option Value      │ Field Option      │ Option Value      │ Field Option      │ Option Value    

## Populate the index

We load the reference transactions, embedding the float32 vector as bytes (the
format Redis stores). The metadata travels alongside each vector so we can
filter on it during search.

In [8]:
def to_records(frame):
    records = []
    for _, row in frame.iterrows():
        records.append({
            "txn_id": row["txn_id"],
            "card_id": row["card_id"],
            "merchant_category": row["merchant_category"],
            "country": row["country"],
            "amount": float(row["amount"]),
            "timestamp": int(row["timestamp"]),
            "is_fraud": int(row["is_fraud"]),
            "vector": row["vector"].tobytes(),
        })
    return records

keys = index.load(to_records(ref_df), id_field="txn_id")
print(f"loaded {len(keys)} transactions into the '{index_name}' index")

loaded 4812 transactions into the 'transactions' index


## Technique 1 — KNN fraud scoring

The core idea: to score a new transaction, find its **k nearest neighbors** in
the indexed history and look at how many of them were fraud. A high
neighborhood fraud rate is a strong, explainable signal.

Let's grab one known-fraud and one known-legit transaction from the unseen test
set and score each.

In [9]:
from redisvl.query import VectorQuery

K = 15

def fraud_score(vector, k=K, filter_expression=None):
    q = VectorQuery(
        vector=vector,
        vector_field_name="vector",
        num_results=k,
        return_fields=["txn_id", "card_id", "merchant_category", "amount", "is_fraud"],
        return_score=True,
        filter_expression=filter_expression,
    )
    neighbors = index.query(q)
    fraud_fraction = np.mean([float(n["is_fraud"]) for n in neighbors])
    return fraud_fraction, neighbors

# one fraud + one legit example from the held-out test set
fraud_i = test_df.index[test_df["is_fraud"] == 1][0]
legit_i = test_df.index[test_df["is_fraud"] == 0][0]

for label, i in [("KNOWN FRAUD", fraud_i), ("KNOWN LEGIT", legit_i)]:
    score, neighbors = fraud_score(test_vectors[i])
    print(f"{label}: neighborhood fraud rate = {score:.0%}  (amount=${test_df.loc[i,'amount']:.2f}, "
          f"category={test_df.loc[i,'merchant_category']})")

KNOWN FRAUD: neighborhood fraud rate = 100%  (amount=$1464.11, category=restaurant)
KNOWN LEGIT: neighborhood fraud rate = 0%  (amount=$65.05, category=online_retail)


The fraudulent transaction sits in a neighborhood dense with other fraud; the
legitimate one is surrounded by normal spend. Here are the actual neighbors
returned for the fraudulent transaction — note the vector distance and the
`is_fraud` flag on each:

In [10]:
score, neighbors = fraud_score(test_vectors[fraud_i])
pd.DataFrame(neighbors)[["txn_id", "card_id", "merchant_category", "amount", "is_fraud", "vector_distance"]]

,txn_id,card_id,merchant_category,amount,is_fraud,vector_distance
0,txn_4325,card_647,fuel,1287.19,1,25.455909729
1,txn_754,card_731,grocery,1416.95,1,28.6217079163
2,txn_2175,card_296,travel,1635.25,1,28.7414302826
3,txn_4213,card_261,fuel,710.18,1,29.5273799896
4,txn_2865,card_52,restaurant,1012.76,1,30.6872444153
5,txn_3738,card_385,electronics,1198.33,1,30.9978752136
6,txn_3267,card_101,fuel,838.94,1,31.6496009827
7,txn_436,card_396,grocery,1076.41,1,31.8088111877
8,txn_3486,card_144,travel,898.66,1,34.2544250488
9,txn_108,card_514,atm_withdrawal,1782.87,1,35.8748664856


### Evaluate the scorer on the full test set

Scoring all 200 unseen transactions lets us pick an alert threshold and see the
precision/recall tradeoff — exactly the knob a fraud team tunes against their
review capacity.

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

scores = np.array([fraud_score(v)[0] for v in test_vectors])
y_true = test_df["is_fraud"].values

THRESHOLD = 0.5  # alert if >= 50% of neighbors are fraud
y_pred = (scores >= THRESHOLD).astype(int)

print(f"Alert threshold: neighborhood fraud rate >= {THRESHOLD:.0%}\n")
print(confusion_matrix(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=["legit", "fraud"], digits=3))

Alert threshold: neighborhood fraud rate >= 50%

[[194   0]
 [  1   5]]

              precision    recall  f1-score   support

       legit      0.995     1.000     0.997       194
       fraud      1.000     0.833     0.909         6

    accuracy                          0.995       200
   macro avg      0.997     0.917     0.953       200
weighted avg      0.995     0.995     0.995       200



## Technique 2 — anomaly detection against normal behavior

KNN scoring needs labeled fraud examples. But genuinely novel fraud may not
resemble any *past* fraud. A complementary, label-light approach is to measure how
far a transaction is from the account's **normal** behavior. If the nearest
*legitimate* transaction is still far away, the transaction is anomalous.

We do this with a [RangeQuery](https://docs.redisvl.com/) filtered to
`is_fraud == 0`. We're asking "are there any normal transactions within distance R?" If
Ther's no match inside the radius ⇒ flagged as anomalous.

In [12]:
from redisvl.query import RangeQuery
from redisvl.query.filter import Num

legit_only = Num("is_fraud") == 0
RADIUS = 3.0  # max L2 distance to be considered "normal"

def anomaly_check(vector, radius=RADIUS):
    q = RangeQuery(
        vector=vector,
        vector_field_name="vector",
        return_fields=["txn_id", "is_fraud"],
        distance_threshold=radius,
        num_results=1,
        filter_expression=legit_only,
    )
    results = index.query(q)
    nearest = results[0]["vector_distance"] if results else None
    is_anomaly = nearest is None or float(nearest) > radius
    return is_anomaly, nearest

for label, i in [("KNOWN FRAUD", fraud_i), ("KNOWN LEGIT", legit_i)]:
    anomaly, nearest = anomaly_check(test_vectors[i])
    near_str = f"{float(nearest):.2f}" if nearest is not None else "none in radius"
    print(f"{label}: nearest normal txn distance = {near_str}  ->  anomaly={anomaly}")

KNOWN FRAUD: nearest normal txn distance = none in radius  ->  anomaly=True
KNOWN LEGIT: nearest normal txn distance = 0.06  ->  anomaly=False


## Technique 3 — combine similarity with rules and velocity

Production fraud systems blend the ML signal with hard business rules and
filters. Because the metadata lives in the same index as the vectors, RedisVL
lets us express these as **filtered vector queries** in a single round trip.

**Example A — scoped KNN:** when investigating a suspicious online purchase,
restrict the neighbor search to the *same merchant category* so the fraud score
reflects peers in that category.

In [13]:
from redisvl.query.filter import Tag

category = test_df.loc[fraud_i, "merchant_category"]

scoped_score, _ = fraud_score(test_vectors[fraud_i], filter_expression=Tag("merchant_category") == category)
global_score, _ = fraud_score(test_vectors[fraud_i])
print(f"category = {category}")
print(f"  fraud score among ALL neighbors:            {global_score:.0%}")
print(f"  fraud score among '{category}' neighbors:   {scoped_score:.0%}")

category = restaurant
  fraud score among ALL neighbors:            100%
  fraud score among 'restaurant' neighbors:   100%


**Example B — velocity check.** A classic fraud rule: too many transactions on
one card in a short window. Here we count a card's transactions in a 1-hour
window using a pure metadata filter query (no vector needed) via `FilterQuery`.

In [14]:
from redisvl.query import FilterQuery

# our injected account-takeover card: a burst of transactions in one hour
busy_card = "card_burst"
window_start = int(ref_df.loc[ref_df["card_id"] == busy_card, "timestamp"].min())
window_end = window_start + 60 * 60  # one hour

velocity_filter = (Tag("card_id") == busy_card) & \
                  (Num("timestamp") >= window_start) & (Num("timestamp") <= window_end)

vq = FilterQuery(
    return_fields=["txn_id", "amount", "merchant_category", "timestamp", "is_fraud"],
    filter_expression=velocity_filter,
    num_results=100,
)
hits = index.query(vq)
print(f"card {busy_card}: {len(hits)} transactions in a 1-hour window -> likely account takeover")
pd.DataFrame(hits)[["txn_id", "amount", "merchant_category", "is_fraud"]] if hits else "no transactions in window"

card card_burst: 12 transactions in a 1-hour window -> likely account takeover


,txn_id,amount,merchant_category,is_fraud
0,txn_burst_0,479.55,travel,1
1,txn_burst_1,1664.99,travel,1
2,txn_burst_2,424.58,electronics,1
3,txn_burst_3,133.18,electronics,1
4,txn_burst_4,616.55,fuel,1
5,txn_burst_5,606.77,online_retail,1
6,txn_burst_6,333.53,atm_withdrawal,1
7,txn_burst_7,1120.52,electronics,1
8,txn_burst_8,40.89,online_retail,1
9,txn_burst_9,576.56,fuel,1


**Example C — high-value online transactions from high-risk countries.** A
pure filter query that surfaces transactions matching a risk policy, ready for
manual review.

In [15]:
review_q = FilterQuery(
    return_fields=["txn_id", "card_id", "amount", "country", "merchant_category", "is_fraud"],
    filter_expression=(Num("amount") >= 500) & ((Tag("country") == "NG") | (Tag("country") == "RU")),
    num_results=10,
)
flagged = index.query(review_q)
print(f"{len(flagged)} high-value transactions from high-risk countries flagged for review")
pd.DataFrame(flagged) if flagged else "none flagged"

10 high-value transactions from high-risk countries flagged for review


,id,txn_id,card_id,amount,country,merchant_category,is_fraud
0,transactions:txn_1731,txn_1731,card_155,1416.87,RU,electronics,1
1,transactions:txn_4875,txn_4875,card_397,970.9,NG,online_retail,1
2,transactions:txn_4128,txn_4128,card_39,1801.74,NG,electronics,1
3,transactions:txn_848,txn_848,card_218,754.6,RU,grocery,1
4,transactions:txn_2959,txn_2959,card_600,2133.8,RU,fuel,1
5,transactions:txn_4960,txn_4960,card_319,1047.54,RU,online_retail,1
6,transactions:txn_535,txn_535,card_478,846.66,NG,restaurant,1
7,transactions:txn_1823,txn_1823,card_5,762.67,RU,online_retail,1
8,transactions:txn_3938,txn_3938,card_547,2172.73,NG,atm_withdrawal,1
9,transactions:txn_2865,txn_2865,card_52,1012.76,NG,restaurant,1


## Putting it together — a real-time scoring function

A single function that a transaction-processing service could call at authorization
time. It returns a decision plus the *reasons*, which is essential for fraud
analysts and for regulatory explainability. All of it is backed by Redis in
milliseconds.

In [16]:
def score_transaction(vector, card_id, timestamp=None):
    reasons = []

    # 1. supervised k-NN signal
    knn_score, _ = fraud_score(vector)
    if knn_score >= 0.5:
        reasons.append(f"{knn_score:.0%} of similar transactions were fraud")

    # 2. unsupervised anomaly signal
    anomaly, nearest = anomaly_check(vector)
    if anomaly:
        reasons.append("unlike any normal transaction (anomalous behavior)")

    # 3. velocity rule
    if timestamp is not None:
        ts = int(timestamp)
        velocity_filter = (Tag("card_id") == card_id) & \
                          (Num("timestamp") >= ts - 60 * 60) & \
                          (Num("timestamp") <= ts)
        recent = index.query(FilterQuery(
            return_fields=["txn_id"],
            filter_expression=velocity_filter,
            num_results=100,
        ))
        if len(recent) >= 10:
            reasons.append(f"high velocity: {len(recent)} transactions on this card")

    decision = "BLOCK" if (knn_score >= 0.5 or anomaly) else "ALLOW"
    return {"decision": decision, "fraud_score": round(float(knn_score), 3), "reasons": reasons}

# score the held-out fraud and legit examples, plus the account-takeover card
burst_vec = ref_df.loc[ref_df["card_id"] == "card_burst", "vector"].iloc[0]
burst_ts = int(ref_df.loc[ref_df["card_id"] == "card_burst", "timestamp"].max())
print("Fraudulent example: ", score_transaction(test_vectors[fraud_i], test_df.loc[fraud_i, "card_id"], test_df.loc[fraud_i, "timestamp"]))
print("Legitimate example: ", score_transaction(test_vectors[legit_i], test_df.loc[legit_i, "card_id"], test_df.loc[legit_i, "timestamp"]))
print("Account takeover:   ", score_transaction(burst_vec, "card_burst", burst_ts))

Fraudulent example:  {'decision': 'BLOCK', 'fraud_score': 1.0, 'reasons': ['100% of similar transactions were fraud', 'unlike any normal transaction (anomalous behavior)']}
Legitimate example:  {'decision': 'ALLOW', 'fraud_score': 0.0, 'reasons': []}
Account takeover:    {'decision': 'BLOCK', 'fraud_score': 1.0, 'reasons': ['100% of similar transactions were fraud', 'unlike any normal transaction (anomalous behavior)', 'high velocity: 12 transactions on this card']}


## Why Redis for fraud detection?

- **Latency** — authorization must complete in tens of milliseconds. Redis serves
  vector KNN, range, and filter queries from memory at that speed.
- **One system, two jobs** — the same Redis index holds the feature vectors *and*
  the transaction metadata, so similarity search and business rules run together
  in a single query instead of joining across systems.
- **Real-time updates** — new labeled transactions are `index.load`-ed and
  immediately searchable; the model's "memory" of fraud stays current without a
  retraining batch job.
- **Velocity & history** — Redis's data structures (counters, sorted sets, TTLs)
  complement vector search for the time-window rules fraud systems rely on.

### Next steps
- Swap the synthetic generator for your own labeled transaction history.
- Replace standardized features with a learned embedding (e.g. an autoencoder or
  a two-tower model) for richer similarity — see the
  [recommendation-systems recipes](../recommendation-systems/).
- Add a [semantic cache](../semantic-cache/) or
  [feature store](../feature-store/) layer for the surrounding pipeline.

### Clean up

In [17]:
# clean up!
index.delete()